# Model 3e+ — species-aware open-set calibration

A self-contained experiment based on Dewang’s **Model 3e** (verified public MAP@5: **0.86**). The complete Model 4c feature pipeline and Model 3e extensions are embedded below, so this notebook does not import files from `Models/Dewang`. Attribution and the original methods are retained.

## Experiment

Model 3e already supports a global EM correction for the share of `new_individual`. This notebook tests whether estimating that share **separately by species** and applying a conservative prior-odds correction improves MAP@5. It compares unadjusted ranking, one global estimate, and per-species estimates over the same animal-disjoint out-of-fold predictions. The correction strength is swept once on those folds, with animal-level bootstrap intervals reported.

This is a validation experiment, not a promise of a 0.88 Kaggle score. The selected setting can still overfit the simulated split. Run the final submission cell only after reviewing the paired validation result.

### Inputs and runtime

Requires the repository’s `Processed_Data/master_dataset.csv`, `miewid_features.npy`, `sift_features.npz`, plus `sample_submission.csv` (or the original competition data directory). The first run computes the Model 3e SIFT cache and generic-keypoint weights; that is the expensive step. Later runs reuse notebook-local cache files.

No external package installation is performed. Required packages match `Models/Dewang/requirements_model4b.txt` plus scikit-learn.

In [ ]:
# Runtime configuration. Set ROOT explicitly if this notebook is copied elsewhere.
from pathlib import Path
import os, time, json, types, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Processed_Data/master_dataset.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Set ROOT to the project directory containing Processed_Data/")
PROC = ROOT / "Processed_Data"
NOTEBOOK_DIR = ROOT / "Models/Emad"
CACHE = NOTEBOOK_DIR / "model3e_plus_cache"
CACHE.mkdir(parents=True, exist_ok=True)

# Full defaults mirror Model 3e. For a smoke run, reduce N_SIMS/SEEDS/ROUNDS
# in this cell, but do not use the resulting score to select a submission.
N_SIMS, N_FOLDS, N_ROUNDS = 24, 5, 3
N_SEEDS, N_TREES, N_WORKERS = 3, 400, max(2, (os.cpu_count() or 2) - 1)
K_CANDIDATES, N_NEIGHBOURS, RANDOM_SEED = 15, 150, 42
RUN_TEST_INFERENCE = False  # switch to True after reviewing validation results

print(f"Project: {ROOT}")
print(f"Cache: {CACHE}")
print(f"Settings: episodes={N_SIMS}, folds={N_FOLDS}, rounds={N_ROUNDS}, seeds={N_SEEDS}")

## Embedded Model 3e implementation

The code below is bundled from `Models/Dewang/model4c_pipeline.py` and `model4e_pipeline.py`. Model 4c is loaded into an in-memory namespace, and Model 3e extends it directly. It does not read or execute a sibling Python file.

In [ ]:
# Full Model 4c core source, embedded from the repository.
import types
c4 = types.ModuleType("model4c_inline")
_MODEL4C_SOURCE = '#!/usr/bin/env python3\n"""\nMODEL 4c (Dewang): Model 4b + variance reduction + extra evidence features.\n\nEverything in Model 4b (0.85373 public) is kept unchanged.  4c adds, in order of\nexpected value / risk:\n\n  1. Seed-bagged LightGBM  (--seeds, default 3).  Each of the three classifiers is\n     trained with several random seeds (different row/column subsamples) and the\n     probabilities are averaged.  Pure variance reduction: cannot add bias.\n  2. More simulated episodes (default 24 vs 16).  The classifiers only ever see\n     simulated open-set episodes, so more/more varied episodes = less overfit to\n     any one simulated gallery composition.\n  3. Softmax / kernel-density identity posterior.  For each query, a temperature-\n     scaled softmax over ALL gallery similarities, summed per identity (a KDE-style\n     class posterior).  Uses every image of an identity rather than just the top 1-2.\n     Also its entropy / max as query-level "how open-set is this query" evidence.\n  4. A second, diverse similarity signal: PCA-whitened embedding (Model 4\'s\n     representation, top 128 components) per-identity max similarity and rank.\n  5. Mild extra LightGBM L2 regularisation (reg_lambda=2) and min_child_samples=40.\n  6. Diagnostics (no effect on predictions):\n       - candidate recall@K on simulated episodes (is the true identity even\n         in the candidate list?  upper bound for MAP@5 on known queries)\n       - sim-vs-test covariate-shift report (which features look different on the\n         real test set than in the simulations)\n\nRules compliance: only the provided MiewID / SIFT features; PCA, LDA, kernel\ndensity / softmax scoring, RANSAC, LightGBM and bagging -- all on the permitted list.\n\nAnti-overfit safeguards (carried over from 4b): identity-disjoint simulated episodes,\nnear-duplicate-aware group holdout, regularised LDA, subsampled LightGBM,\nout-of-fold stacking.\n\nThe SIFT pair cache is shared with Model 4b (same matching parameters), so a\nprevious 4b run means no SIFT recomputation.\n\nRun:  python model4c_pipeline.py                   (validation only)\n      python model4c_pipeline.py --predict-test     (also writes submission)\n      python model4c_pipeline.py --seeds 1 --sims 16   (fast, ~4b-equivalent)\n"""\nimport argparse\nimport json\nimport os\nimport re\nimport time\nimport zipfile as _zipfile\nfrom concurrent.futures import ThreadPoolExecutor\nfrom functools import lru_cache\nfrom threading import Lock\n\nimport cv2\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom scipy.sparse.csgraph import connected_components\nfrom sklearn.decomposition import PCA\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA\n\nNEW = "new_individual"\nVIEWS = ["left", "right", "front", "back", "frontleft", "frontright",\n         "backleft", "backright", "up", "down"]\nFLANK = np.array([0, 1, 2, 2, 0, 1, 0, 1, 2, 2])  # left-ish, right-ish, other\n\n# Mutable globals set by load_data / sift_matrices\nX = None          # (N, D) L2-normalised MiewID features\nIDS = None        # (N,) image ids\nY = None          # (N,) identity labels (None for test)\nSP = None         # (N,) species\nVIEW = None       # (N,) viewpoint code (int)\nGOOD = None       # sparse (N, N) SIFT ratio-test matches\nINL = None        # sparse (N, N) RANSAC inliers\n\n# Hyperparams set at runtime\nK = 15                                               # candidates per query\nN_SIMS = 24                                          # simulated episodes\nFOLDS = 4                                            # stacking folds\nROUNDS = 3                                           # stacking rounds\nSEEDS = 3                                            # LightGBM bagging seeds\nFRAC = (0.3, 0.5)                                    # simulated query-holdout fraction range\nTAU = 0.05                                           # softmax temperature (posterior)\nWHITEN_COMPS = 128                                   # PCA comps for whitened signal\nNEIGHBOURS = 150                                     # SIFT neighbourhood size\nMATCH_RATIO = 0.85                                   # SIFT ratio test\nRANSAC_PX = 12.0                                     # RANSAC threshold\nSIFT_PULL = 0.05                                     # SIFT weight in candidate selection\nREGIMES = [None, (0.95, 100), (0.99, 80), (0.9, 60)] # near-duplicate linking\nPARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=31,\n              min_child_samples=40, reg_lambda=2.0, subsample=0.8, subsample_freq=1,\n              colsample_bytree=0.8, n_estimators=400, verbose=-1)\nINDEX_COLS = re.compile(r"^(nb|nw|sb|sw)\\d$")\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# Data loading\n# ═══════════════════════════════════════════════════════════════════════════════\ndef unit(a):\n    return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)\n\n\ndef load_data(root):\n    """Load master_dataset.csv and miewid_features.npy, populate globals."""\n    global X, IDS, Y, SP, VIEW\n    proc = os.path.join(root, "Processed_Data")\n    master = pd.read_csv(os.path.join(proc, "master_dataset.csv"))\n    raw = np.load(os.path.join(proc, "miewid_features.npy"))\n    need = {"id", "identity", "species", "viewpoint", "split"}\n    missing_cols = need - set(master.columns)\n    if missing_cols:\n        raise SystemExit(f"master_dataset.csv is missing columns {sorted(missing_cols)}; "\n                         f"found {list(master.columns)}")\n    # Align features with master_dataset order via miewid_index (fallback: row order)\n    if "miewid_index" in master.columns:\n        feat_idx = master["miewid_index"].to_numpy()\n    else:\n        print("  WARNING: no \'miewid_index\' column; assuming features are in row order")\n        feat_idx = np.arange(len(master))\n    X = unit(np.asarray(raw[feat_idx], dtype=np.float32))\n    IDS = master["id"].to_numpy()\n    Y = master["identity"].to_numpy(dtype=object)\n    SP = master["species"].to_numpy(dtype=object)\n    VIEW = np.array([{v: i for i, v in enumerate(VIEWS)}.get(str(v), -1)\n                      for v in master["viewpoint"]], dtype=int)\n    print(f"  species values: {sorted(set(map(str, SP)))}   splits: "\n          f"{master[\'split\'].value_counts().to_dict()}")\n    train = master[master["split"] == "train"].reset_index(drop=True)\n    test = master[master["split"] == "test"].reset_index(drop=True)\n    # Map to global indices in X\n    id_to_idx = {iid: i for i, iid in enumerate(IDS)}\n    train_idx = np.array([id_to_idx[i] for i in train["id"]])\n    test_idx = np.array([id_to_idx[i] for i in test["id"]])\n    return train, test, master, train_idx, test_idx\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# SIFT verification  (ThreadPoolExecutor + lru_cache — no fork, no zip issues)\n# ═══════════════════════════════════════════════════════════════════════════════\nclass SiftCache:\n    """Thread-safe SIFT descriptor loader with LRU cache.\n\n    OpenCV BFMatcher / estimateAffinePartial2D release the GIL, so\n    ThreadPoolExecutor gives real parallelism without fork-related NPZ\n    issues that multiprocessing.Pool triggers on Python 3.10+.\n    """\n\n    def __init__(self, npz_path, cache_path, workers=2, mem_items=2000):\n        from pathlib import Path\n        extracted = Path(npz_path).with_suffix("")\n        self.extracted = extracted if extracted.is_dir() else None\n        self.npz = None if self.extracted else np.load(npz_path)\n        self.npz_keys = ({p.stem for p in self.extracted.glob("*.npy")} if self.extracted else set(self.npz.files))\n        hits = sum(f"{i}_desc" in self.npz_keys for i in IDS)\n        print(f"  SIFT npz: {len(self.npz_keys)} arrays, descriptors found for "\n              f"{hits}/{len(IDS)} images", flush=True)\n        if hits == 0:\n            raise SystemExit(f"No \'<id>_desc\' keys in {npz_path}. Example keys: "\n                             f"{list(self.npz_keys)[:6]}")\n        self.read_lock = Lock()          # NpzFile\'s internal ZipFile is not thread-safe\n        self.workers = workers\n        self.cache_path = cache_path\n        self.done = {}\n        if os.path.exists(cache_path):\n            with np.load(cache_path) as z:\n                for p, v in zip(z["pairs"], z["values"]):\n                    self.done[tuple(p)] = tuple(v)\n        # Bind lru_cache *per instance* so it can be garbage-collected\n        self._descriptors = lru_cache(maxsize=mem_items)(self._descriptors_raw)\n\n    def _descriptors_raw(self, i):\n        """Load RootSIFT descriptors and keypoint positions for one image."""\n        iid = IDS[i]\n        dkey, xykey = f"{iid}_desc", f"{iid}_xy"\n        if dkey not in self.npz_keys:\n            return np.zeros((0, 128), np.float32), np.zeros((0, 2), np.float32)\n        with self.read_lock:\n            d = np.asarray((np.load(self.extracted / (dkey + ".npy")) if self.extracted else self.npz[dkey]), dtype=np.float32)\n            xy = np.asarray((np.load(self.extracted / (xykey + ".npy")) if self.extracted else self.npz[xykey]), dtype=np.float32)[:, :2]\n        d = np.sqrt(d / np.maximum(d.sum(1, keepdims=True), 1e-12))\n        return np.ascontiguousarray(d), xy\n\n    def verify(self, pair):\n        """Mutual ratio-test matches and RANSAC inliers between two images."""\n        a, b = pair\n        da, xa = self._descriptors(a)\n        db, xb = self._descriptors(b)\n        if len(da) < 4 or len(db) < 4:\n            return 0, 0\n        bf = cv2.BFMatcher()\n        back = np.array([m.trainIdx for m in bf.match(db, da)])\n        good = [m[0] for m in bf.knnMatch(da, db, k=2)\n                if len(m) == 2 and m[0].distance < MATCH_RATIO * m[1].distance\n                and back[m[0].trainIdx] == m[0].queryIdx]\n        if len(good) < 4:\n            return len(good), 0\n        src = np.float32([xa[m.queryIdx] for m in good])\n        dst = np.float32([xb[m.trainIdx] for m in good])\n        # Deterministic RANSAC per pair\n        cv2.setRNGSeed(int((int(a) * 100003 + int(b)) % (2**31 - 1)))\n        _, inliers = cv2.estimateAffinePartial2D(\n            src, dst, method=cv2.RANSAC, ransacReprojThreshold=RANSAC_PX)\n        return len(good), 0 if inliers is None else int(inliers.sum())\n\n    def save(self):\n        os.makedirs(os.path.dirname(self.cache_path) or ".", exist_ok=True)\n        pairs = sorted(self.done)\n        np.savez_compressed(\n            self.cache_path,\n            pairs=np.array(pairs, dtype=np.int64).reshape(-1, 2),\n            values=np.array([self.done[p] for p in pairs]).reshape(-1, 2))\n\n\ndef sift_matrices(npz_path, cache_path, workers=2):\n    """Build symmetric sparse matrices of SIFT matches/inliers for all needed pairs."""\n    global GOOD, INL\n    cache = SiftCache(npz_path, cache_path, workers)\n    n = len(X)\n    # Find pairs to verify: nearest same-species neighbours\n    wanted = set()\n    for sp in ("hyena", "leopard"):\n        ix = np.where(SP == sp)[0]\n        # Chunk the similarity computation to limit memory\n        for start in range(0, len(ix), 256):\n            q = ix[start:start + 256]\n            sim = X[q] @ X[ix].T\n            # Zero out self-similarities\n            for local_i, global_i in enumerate(q):\n                col = np.searchsorted(ix, global_i)\n                if col < sim.shape[1] and ix[col] == global_i:\n                    sim[local_i, col] = -9.0\n            k = min(NEIGHBOURS, len(ix) - 1)\n            nb = np.argsort(-sim, 1)[:, :k]\n            for row, nbs in zip(q, nb):\n                wanted.update((min(int(row), int(j)), max(int(row), int(j)))\n                              for j in ix[nbs])\n    missing = sorted(wanted - cache.done.keys())\n    print(f"  SIFT: {len(wanted)} pairs needed, {len(missing)} uncached", flush=True)\n    if missing:\n        cv2.setNumThreads(1)   # each thread does its own matching\n        t0 = time.time()\n        BLOCK = 20000\n        with ThreadPoolExecutor(max_workers=workers) as pool:\n            for s0 in range(0, len(missing), BLOCK):\n                block = missing[s0:s0 + BLOCK]\n                for pair, value in zip(block, pool.map(cache.verify, block)):\n                    cache.done[pair] = value\n                cache.save()      # re-running resumes from here if interrupted\n                print(f"    verified {s0 + len(block)}/{len(missing)}  "\n                      f"({time.time() - t0:.0f}s)", flush=True)\n        cache.save()\n    # Build sparse matrices from wanted pairs only\n    pp = np.array(sorted(wanted), dtype=int).reshape(-1, 2)\n    vv = np.array([cache.done[tuple(p)] for p in pp]).reshape(-1, 2)\n    a, b = pp.T\n    def mat(v):\n        return sparse.coo_matrix(\n            (np.r_[v, v], (np.r_[a, b], np.r_[b, a])), shape=(n, n)).tocsr()\n    GOOD, INL = mat(vv[:, 0]), mat(vv[:, 1])\n    print(f"  SIFT matrices built: {GOOD.nnz} nonzeros", flush=True)\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# PCA + LDA embedding  (upgrade #1 over Model 4)\n# ═══════════════════════════════════════════════════════════════════════════════\ndef embed(gallery_idx, query_idx, labels):\n    """PCA -> LDA projection (+ a PCA-whitened view).  PCA is unsupervised (sees\n    queries too).  LDA is supervised on gallery labels only -> identity-disjoint."""\n    all_idx = np.r_[gallery_idx, query_idx]\n    n_comp = min(384, len(all_idx) - 1, X.shape[1])\n    pca = PCA(n_comp).fit(X[all_idx])\n    pg, pq = pca.transform(X[gallery_idx]), pca.transform(X[query_idx])\n    lda = LDA(solver="eigen", shrinkage=0.3).fit(pg, labels)\n    w = min(WHITEN_COMPS, n_comp)\n    scale = 1.0 / np.sqrt(pca.explained_variance_[:w] + 1e-9)\n    return (unit(lda.transform(pg)), unit(lda.transform(pq)),\n            unit(pg[:, :w] * scale), unit(pq[:, :w] * scale))\n\n\ndef posterior(sim, inv, n, tau=None):\n    """Softmax over all gallery images, summed per identity (KDE-style posterior)."""\n    tau = TAU if tau is None else tau\n    e = np.exp((sim - sim.max(1, keepdims=True)) / tau)\n    onehot = sparse.csr_matrix((np.ones(len(inv)), (np.arange(len(inv)), inv)),\n                               shape=(len(inv), n))\n    num = np.asarray(onehot.T.dot(e.T)).T\n    return num / np.maximum(num.sum(1, keepdims=True), 1e-12)\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# Feature extraction  (upgrade #4 — richer features)\n# ═══════════════════════════════════════════════════════════════════════════════\ndef per_identity(sim, inv, n):\n    """Per query: max and mean-of-top-2 similarity to each identity\'s images."""\n    best = np.empty((sim.shape[0], n))\n    top2 = np.empty((sim.shape[0], n))\n    for j in range(n):\n        s = sim[:, inv == j]\n        best[:, j] = s.max(1)\n        top2[:, j] = np.sort(s, 1)[:, -2:].mean(1)\n    return best, top2\n\n\ndef duplicate_groups(ix, link):\n    """Group near-duplicate images of one identity."""\n    if link is None:\n        return np.arange(len(ix))\n    edges = (X[ix] @ X[ix].T > link[0]) | (INL[ix][:, ix].toarray() >= link[1])\n    return connected_components(sparse.csr_matrix(edges))[1]\n\n\ndef species_features(gi, qi, sp):\n    """Build query-level and (query, candidate) feature DataFrames for one species."""\n    Rg, Rq = X[gi], X[qi]\n    yg, vg, vq = Y[gi], VIEW[gi], VIEW[qi]\n    Lg, Lq, Wg, Wq = embed(gi, qi, yg)\n\n    ids, inv = np.unique(yg, return_inverse=True)\n    n, nq = len(ids), len(qi)\n    if n < K:\n        raise ValueError(f"only {n} gallery identities for {sp}, need >= candidates ({K})")\n\n    sim_l, sim_r = Lq @ Lg.T, Rq @ Rg.T\n    lda1, lda2 = per_identity(sim_l, inv, n)\n    raw1, raw2 = per_identity(sim_r, inv, n)\n    sim_w = Wq @ Wg.T\n    wht1, _ = per_identity(sim_w, inv, n)\n    post_l, post_r = posterior(sim_l, inv, n), posterior(sim_r, inv, n)\n    ent_l = -(post_l * np.log(np.maximum(post_l, 1e-12))).sum(1) / np.log(max(n, 2))\n    pmax_l, pmax_r = post_l.max(1), post_r.max(1)\n    centroid = unit(np.stack([Lg[inv == j].mean(0) for j in range(n)]))\n    cen = Lq @ centroid.T\n\n    # SIFT per-identity features\n    inl_g = INL[qi][:, gi].toarray().astype(np.float32)\n    good_g = GOOD[qi][:, gi].toarray().astype(np.float32)\n    sift1, sift2 = per_identity(inl_g, inv, n)\n    good1, _ = per_identity(good_g, inv, n)\n    # Support: how many gallery images of each identity have ≥7 RANSAC inliers\n    support = (inl_g >= 7).astype(np.float32) @ np.eye(n, dtype=np.float32)[inv]\n\n    # Same-viewpoint similarity (upgrade #4)\n    same_view = np.full((nq, n), np.nan)\n    for j in range(n):\n        for v in np.unique(vq):\n            cols = np.where((inv == j) & (vg == v))[0]\n            if len(cols):\n                same_view[vq == v, j] = sim_l[np.ix_(vq == v, cols)].max(1)\n\n    # Viewpoint coverage per identity\n    by_view = np.stack([np.bincount(inv[vg == v], minlength=n) for v in range(len(VIEWS))], 1)\n    by_flank = np.stack([by_view[:, FLANK == f].sum(1) for f in range(3)], 1)\n    seen = vq >= 0\n    n_view = np.where(seen[:, None], by_view[:, vq.clip(0)].T, np.nan)\n    n_flank = np.where(seen[:, None], by_flank[:, FLANK[vq.clip(0)]].T, np.nan)\n\n    # Transductive: nearest query neighbours by embedding and SIFT\n    qq_r, qq_l = Rq @ Rq.T, Lq @ Lq.T\n    qq_s = INL[qi][:, qi].toarray().astype(np.float32)\n    np.fill_diagonal(qq_r, -1)\n    np.fill_diagonal(qq_l, -1)\n    nb = np.argsort(-qq_l, 1)[:, :5]\n    sb = np.argsort(-qq_s, 1)[:, :5]\n    nbs = np.take_along_axis(qq_l, nb, 1)\n    sbs = np.take_along_axis(qq_s, sb, 1)\n\n    t_l = np.take_along_axis(lda1, np.argsort(-lda1, 1)[:, :3], 1)\n    t_r = -np.sort(-raw1, 1)[:, :3]\n    t_s = -np.sort(-sift1, 1)[:, :3]\n    g_top = -np.sort(-sim_l, 1)\n\n    # Query-level features\n    q = pd.DataFrame({\n        "sp": int(sp == "hyena"), "view": vq, "ng": len(gi),\n        "l1": t_l[:, 0], "l2": t_l[:, 1], "l3": t_l[:, 2],\n        "r1": t_r[:, 0], "r2": t_r[:, 1], "r3": t_r[:, 2],\n        "lm12": t_l[:, 0] - t_l[:, 1], "rm12": t_r[:, 0] - t_r[:, 1],\n        "gd5": g_top[:, :5].mean(1), "gd20": g_top[:, :20].mean(1),\n        "pmax_l": pmax_l, "pmax_r": pmax_r, "ent_l": ent_l,\n        "w1": wht1.max(1),\n        "qq1": nbs[:, 0], "qq3": nbs[:, :3].mean(1), "qq5": nbs.mean(1),\n        "qqr1": qq_r.max(1),\n        "qn6": (qq_r > .6).sum(1), "qn7": (qq_r > .7).sum(1), "qn8": (qq_r > .8).sum(1),\n        "qq_minus_g": nbs[:, 0] - t_l[:, 0], "nb_g1": t_l[nb[:, 0], 0],\n        "s1": t_s[:, 0], "s2": t_s[:, 1], "s3": t_s[:, 2], "sg": good_g.max(1),\n        "qs1": sbs[:, 0], "qs3": sbs[:, :3].mean(1),\n        "qsn10": (qq_s >= 10).sum(1), "qsn20": (qq_s >= 20).sum(1),\n        "qs_minus_s": sbs[:, 0] - t_s[:, 0], "nbs_s1": t_s[sb[:, 0], 0],\n    })\n    for k in range(5):\n        q[f"nb{k}"], q[f"nw{k}"] = nb[:, k], nbs[:, k]\n        q[f"sb{k}"], q[f"sw{k}"] = sb[:, k], sbs[:, k]\n\n    # Candidate-level features (upgrade #4)\n    cand = np.argsort(-(lda1 + SIFT_PULL * np.tanh(sift1 / 15)), 1)[:, :K]\n    r, c = np.repeat(np.arange(nq), K), cand.ravel()\n    nb_ev = np.max(np.minimum(nbs[:, :, None],\n                               np.stack([lda1[nb[:, k]] for k in range(5)], 1)), 1)\n    sb_ev = np.max(np.minimum(sbs[:, :, None],\n                               np.stack([sift1[sb[:, k]] for k in range(5)], 1)), 1)\n    rank_of = lambda m: (-m).argsort(1).argsort(1)[r, c]\n\n    cd = pd.DataFrame({\n        "q": r, "c": c, "rank": np.tile(np.arange(K), nq), "ident": ids[c],\n        "mxr": raw1[r, c], "m2r": raw2[r, c],\n        "mxl": lda1[r, c], "m2l": lda2[r, c], "cl": cen[r, c],\n        "gapl": lda1[r, c] - t_l[r, 0], "gapr": raw1[r, c] - t_r[r, 0],\n        "same": same_view[r, c],\n        "cnt": np.bincount(inv)[c], "nbev": nb_ev[r, c], "sbev": sb_ev[r, c],\n        "rk_r": rank_of(raw1), "rk_c": rank_of(cen), "rk_s": rank_of(sift1),\n        "n_view": n_view[r, c], "n_flank": n_flank[r, c],\n        "support": support[r, c],\n        "sx": sift1[r, c], "s2x": sift2[r, c], "gx": good1[r, c],\n        "sgap": sift1[r, c] - t_s[r, 0],\n        "pl": post_l[r, c], "pr": post_r[r, c],\n        "pgap": pmax_l[r] - post_l[r, c],\n        "mxw": wht1[r, c], "rk_w": rank_of(wht1),\n    })\n    return q, cd\n\n\ndef build_frames(gi, qi):\n    """Build feature frames across both species."""\n    qs, cs, order, off = [], [], [], 0\n    for sp in ("hyena", "leopard"):\n        g = gi[SP[gi] == sp]\n        qq = qi[SP[qi] == sp]\n        q, c = species_features(g, qq, sp)\n        c["q"] += off\n        q[[f"{p}{k}" for p in ("nb", "sb") for k in range(5)]] += off\n        qs.append(q)\n        cs.append(c)\n        order.append(qq)\n        off += len(qq)\n    return dict(q=pd.concat(qs, ignore_index=True),\n                c=pd.concat(cs, ignore_index=True),\n                order=np.concatenate(order))\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# Simulated open-set episodes  (upgrade #5 — regime-diverse)\n# ═══════════════════════════════════════════════════════════════════════════════\ndef simulate(seed, train_idx):\n    """Identity-disjoint open-set split with regime-mixture simulation."""\n    rng = np.random.default_rng(seed)\n    link = REGIMES[seed % len(REGIMES)]\n    gallery, known, unknown = [], [], []\n    for sp in ("hyena", "leopard"):\n        p_unknown = rng.uniform(0.2, 0.35)\n        frac = rng.uniform(*FRAC)\n        cap = int(rng.integers(12, 33))\n        ix = train_idx[SP[train_idx] == sp]\n        names = np.array(sorted(set(Y[ix])))\n        rng.shuffle(names)\n        new_set = set(names[:int(round(p_unknown * len(names)))])\n        for name in names:\n            rows = ix[Y[ix] == name].copy()\n            rng.shuffle(rows)\n            if name in new_set:\n                unknown.extend(rows[:cap].tolist())\n            elif len(rows) < 2:\n                gallery.extend(rows.tolist())\n            else:\n                n_q = min(max(int(round(frac * len(rows))), 1), cap, len(rows) - 1)\n                groups = duplicate_groups(rows, link)\n                held = []\n                for c in rng.permutation(np.unique(groups)):\n                    members = rows[groups == c].tolist()\n                    if len(held) < n_q and len(held) + len(members) < len(rows):\n                        held.extend(members)\n                known.extend(held[:cap])\n                held_set = set(held)\n                gallery.extend(i for i in rows if i not in held_set)\n    gi = np.array(gallery, dtype=int)\n    qi = np.array(known + unknown, dtype=int)\n    frames = build_frames(gi, qi)\n    truth = np.where(np.isin(frames["order"], unknown), NEW, Y[frames["order"]])\n    frames["truth"] = truth\n    frames["q"]["unk"] = (truth == NEW).astype(int)\n    frames["c"]["y"] = (frames["c"].ident.to_numpy(dtype=object) ==\n                        truth[frames["c"].q.to_numpy()]).astype(int)\n    return frames\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# LightGBM stacked classifiers  (upgrade #2 — replaces logistic regression)\n# ═══════════════════════════════════════════════════════════════════════════════\ndef columns(s):\n    # "ng" (gallery size) is excluded: it is a per-episode fingerprint in the sims and\n    # far outside the simulated range at test time (gallery = full train).\n    qf = [c for c in s["q"].columns\n          if c not in ("unk", "ng") and not INDEX_COLS.match(c)]\n    cf = [c for c in s["c"].columns if c not in ("q", "c", "ident", "y")]\n    return qf, cf\n\n\ndef candidate_matrix(s, qf, cf):\n    return pd.concat([s["c"][cf].reset_index(drop=True),\n                       s["q"].iloc[s["c"].q.to_numpy()][qf].reset_index(drop=True)], axis=1)\n\n\nclass Bag:\n    """Average predict_proba over several LightGBM models (different seeds)."""\n\n    def __init__(self, models):\n        self.models = models\n\n    def predict_proba(self, x):\n        return np.mean([m.predict_proba(x) for m in self.models], axis=0)\n\n\ndef bagged(params, x, y):\n    base = params.get("random_state", 0) or 0\n    return Bag([lgb.LGBMClassifier(**{**params, "random_state": base + 101 * k}).fit(x, y)\n                for k in range(SEEDS)])\n\n\ndef fit(sims):\n    """Fit new-individual detector, closed-set candidate scorer, all-query scorer\n    (each a bag of SEEDS LightGBM models)."""\n    qf, cf = columns(sims[0])\n    known = lambda s: s["q"].unk.to_numpy()[s["c"].q.to_numpy()] == 0\n    cx = pd.concat([candidate_matrix(s, qf, cf) for s in sims])\n    cy = np.concatenate([s["c"].y.to_numpy() for s in sims])\n    ck = np.concatenate([known(s) for s in sims])\n    detector = bagged({**PARAMS, "n_estimators": 300, "num_leaves": 15},\n                      pd.concat([s["q"][qf] for s in sims]),\n                      np.concatenate([s["q"].unk.to_numpy() for s in sims]))\n    closed_scorer = bagged(PARAMS, cx[ck], cy[ck])\n    all_scorer = bagged(PARAMS, cx, cy)\n    return detector, closed_scorer, all_scorer, (qf, cf)\n\n\ndef predict(models, s):\n    detector, closed, anyq, (qf, cf) = models\n    nq = len(s["q"])\n    cx = candidate_matrix(s, qf, cf)\n    p_new = detector.predict_proba(s["q"][qf])[:, 1]\n    raw = anyq.predict_proba(cx)[:, 1].reshape(nq, K)\n    within = closed.predict_proba(cx)[:, 1].reshape(nq, K)\n    within /= np.maximum(within.sum(1, keepdims=True), 1e-9)\n    return dict(pn=p_new, raw=raw, pc=within,\n                pb=(p_new + np.clip(1 - raw.sum(1), 0, 1)) / 2)\n\n\ndef rank(s, r):\n    """Top-5 labels: P(new) vs (1 - P(new)) * P(identity | known)."""\n    nq = len(s["q"])\n    labels = np.concatenate([\n        s["c"].ident.to_numpy(dtype=object).reshape(nq, K),\n        np.full((nq, 1), NEW, dtype=object)], 1)\n    scores = np.concatenate([(1 - r["pb"])[:, None] * r["pc"],\n                              r["pb"][:, None]], 1)\n    return np.take_along_axis(labels, np.argsort(-scores, 1)[:, :5], 1)\n\n\ndef candidate_recall(s):\n    """Fraction of known-identity queries whose true identity is among the K candidates\n    (hard upper bound on known-query MAP@5)."""\n    nq = len(s["q"])\n    cand = s["c"].ident.to_numpy(dtype=object).reshape(nq, K)\n    known = s["truth"] != NEW\n    hit = (cand == s["truth"][:, None]).any(1)\n    return float(hit[known].mean()) if known.any() else float("nan")\n\n\ndef shift_report(sims, final, top=8):\n    """Compare query-level features of the real test frame with the simulations.\n    Big standardised shifts = the classifiers see test data unlike their training."""\n    cols = [c for c in final["q"].columns\n            if c in sims[0]["q"].columns and c not in ("unk", "ng")\n            and not INDEX_COLS.match(c)]\n    a = pd.concat([s["q"][cols] for s in sims]).astype(float)\n    b = final["q"][cols].astype(float)\n    sd = a.std().replace(0, np.nan)\n    z = ((b.median() - a.median()) / sd).dropna()\n    z = z.reindex(z.abs().sort_values(ascending=False).index)\n    print("  covariate shift (test median - sim median) / sim std, largest first:")\n    for name, v in z.head(top).items():\n        print(f"    {name:10s} {v:+.2f}   (sim {a[name].median():.3f}  test {b[name].median():.3f})")\n    if z.abs().head(top).max() > 1.0:\n        print("    -> |shift| > 1 on some features: consider widening simulate() regimes")\n\n\ndef map5(preds, truth):\n    return np.mean([next((1 / (i + 1) for i, p in enumerate(row) if p == t), 0.0)\n                    for row, t in zip(preds, truth)])\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# Transductive stacking  (upgrade #3)\n# ═══════════════════════════════════════════════════════════════════════════════\ndef augment(s, r):\n    """Add previous round\'s predictions as features for the next round."""\n    q, c, nq = s["q"].copy(), s["c"].copy(), len(s["q"])\n    codes = pd.factorize(c.ident)[0].reshape(nq, K)\n    q["r_pn"] = r["pn"]\n    q["r_pb"] = r["pb"]\n    q["r_rawsum"] = r["raw"].sum(1)\n    q["r_rawmax"] = r["raw"].max(1)\n    c["r_raw"] = r["raw"].ravel()\n    c["r_pc"] = r["pc"].ravel()\n    for kind, weight, scale in (("nb", "nw", 1.0), ("sb", "sw", 30.0)):\n        evidence = np.zeros((nq, K))\n        support = np.zeros((nq, K))\n        pb_list, w_list, top_list = [], [], []\n        for k in range(5):\n            idx = q[f"{kind}{k}"].to_numpy()\n            sim = q[f"{weight}{k}"].to_numpy() / scale\n            wk = (np.clip(sim - 0.4, 0, None) if kind == "nb"\n                  else np.clip(sim, 0, 1))\n            agree = ((codes[idx][:, None, :] == codes[:, :, None]) *\n                     r["raw"][idx][:, None, :]).sum(2)\n            evidence = np.maximum(evidence,\n                                   np.minimum(agree, np.clip(sim, 0, 1)[:, None]))\n            support += wk[:, None] * agree\n            pb_list.append(r["pb"][idx])\n            w_list.append(wk)\n            top_list.append(r["raw"][idx].max(1))\n        pb = np.stack(pb_list, 1)\n        w = np.stack(w_list, 1)\n        q[f"{kind}_pb0"] = pb[:, 0]\n        q[f"{kind}_pbw"] = (pb * w).sum(1) / np.maximum(w.sum(1), 1e-6)\n        q[f"{kind}_pbmin"] = np.where(w > 0, pb, 1).min(1)\n        q[f"{kind}_w"] = w.sum(1)\n        q[f"{kind}_rm0"] = top_list[0]\n        c[f"r_{kind}ev"] = evidence.ravel()\n        c[f"r_{kind}sum"] = support.ravel()\n    return dict(s, q=q, c=c)\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# Validation & submission\n# ═══════════════════════════════════════════════════════════════════════════════\ndef validate_submission(sub, test_df, train_df, sample_path):\n    assert len(sub) == len(test_df), "row count mismatch"\n    assert sub["id"].is_unique, "duplicate ids"\n    assert set(sub["id"]) == set(test_df["id"]), "ids differ from test ids"\n    if sample_path and os.path.exists(sample_path):\n        sample = pd.read_csv(sample_path)\n        assert list(sample.columns) == list(sub.columns), \\\n            f"columns {list(sub.columns)} != sample {list(sample.columns)}"\n        assert (sub["id"].values == sample["id"].values).all(), \\\n            "id order differs from sample_submission"\n    else:\n        print("  (sample_submission.csv not found: only internal checks done)")\n    known = {sp: set(train_df[train_df["species"] == sp]["identity"])\n             for sp in ("hyena", "leopard")}\n    sp_of = dict(zip(test_df["id"], test_df["species"]))\n    for i, pr in zip(sub["id"], sub["predictions"]):\n        t = pr.split()\n        assert len(t) == 5 and len(set(t)) == 5, f"bad row {i}"\n        for x in t:\n            assert x == NEW or x in known[sp_of[i]], \\\n                f"wrong-species/unknown label {x} in {i}"\n    print("  submission checks passed")\n\n\n'
exec(compile(_MODEL4C_SOURCE, "<embedded Model 4c>", "exec"), c4.__dict__)

# Model 3e methods and extensions, embedded; main()/CLI block is omitted.
_MODEL3E_SOURCE = '#!/usr/bin/env python3\n"""\nMODEL 3e: Model 4c + the gaps listed in the team\'s improvement note (26 Nov).\nKeep model4c_pipeline.py in the same folder.\n\n  1. Test mismatch      simulated queries follow the test viewpoint mix; a test-vs-sim\n                        classifier measures the remaining gap and gives importance\n                        weights, used only if they help weighted validation.\n  2. Validation         folds are split by ANIMAL, not by simulated episode.\n  3. One ranker         LightGBM lambdarank over K candidates + a new_individual row;\n                        compared with 4c\'s hand-made fusion and a 50/50 blend.\n  4. Background SIFT    generic keypoints (similar to descriptors in many images) are\n                        down-weighted.\n  5. More SIFT          weighted inliers, inlier hull area, rotation/scale\n                        consistency, affine-model inliers.\n  6. PLDA               two-covariance likelihood ratio per (query, identity).\n\nNo leaderboard probing: every choice is made on training-set validation and one\nsubmission file is written.\n\nRules: provided features only; numpy / scipy / scikit-learn / LightGBM / OpenCV\nmatching.  Item 1 uses UNLABELLED test features and metadata: confirm with the TA\nteam, or pass --no-test-adapt.\n\nValidation caveat: neighbour-stacking features still come from first-level models\nthat saw other folds (ordinary stacking, not nested), so scores stay slightly\noptimistic.  They are not comparable with 4c\'s episode-split numbers.\n\nRun:  python model3e_pipeline.py                  validation only\n      python model3e_pipeline.py --predict-test   validation + submission\n      add --retune to search LightGBM size/regularisation on the animal split\n      add --quick for one seed (fast first look)\nThe first run recomputes SIFT pairs (new statistics, new cache file).\n"""\nimport argparse\nimport json\nimport os\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\n\nimport cv2\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom scipy.linalg import eigh\nfrom sklearn.decomposition import PCA\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import GroupKFold\n\n# Model 4c runtime is embedded in the previous cell as c4\n\nNEW = c4.NEW\nSPECIES = ("hyena", "leopard")\nEXTRA = {}                  # name -> sparse (N, N) matrix of extra SIFT statistics\nEXTRA_NAMES = ["wsift", "hull", "rig", "aff"]\nEXTRA_PREFIX = tuple(f"{n}_" for n in EXTRA_NAMES + ["plda"])\n_base_species_features = c4.species_features\n\n\n# ═══ SIFT: keypoint distinctiveness + richer pair statistics ═════════════════\nclass SiftCache3e(c4.SiftCache):\n    """4c\'s matcher (same good/inlier counts) with extra per-pair statistics."""\n\n    def __init__(self, npz_path, cache_path, workers):\n        super().__init__(npz_path, cache_path, workers, mem_items=2000)\n        self.weights = {}\n\n    def genericness(self, train_idx, path, n_ref=200, per_ref=100, top=10):\n        """For each keypoint: mean similarity to its closest descriptor in each of the\n        `top` most similar reference TRAIN images.  High = common texture."""\n        g = {}\n        if os.path.exists(path):\n            with np.load(path) as z:\n                g = {int(k): z[k] for k in z.files}\n        rng = np.random.default_rng(0)\n        for sp in SPECIES:\n            todo = [i for i in np.where(c4.SP == sp)[0] if i not in g]\n            if not todo:\n                continue\n            ref_ids, blocks = [], []\n            for i in rng.permutation(train_idx[c4.SP[train_idx] == sp]):\n                d = c4.SiftCache._descriptors_raw(self, int(i))[0]\n                if len(d):\n                    ref_ids.append(int(i))\n                    blocks.append(d[rng.choice(len(d), per_ref, replace=len(d) < per_ref)])\n                if len(ref_ids) == n_ref:\n                    break\n            ref = np.concatenate(blocks)\n            column = {i: j for j, i in enumerate(ref_ids)}\n            print(f"  keypoint weights, {sp}: {len(todo)} images vs "\n                  f"{len(ref_ids)} reference images", flush=True)\n            for n, i in enumerate(todo, 1):\n                d = c4.SiftCache._descriptors_raw(self, int(i))[0]\n                out = np.zeros(len(d), np.float32)\n                for s in range(0, len(d), 512):\n                    sim = d[s:s + 512] @ ref.T\n                    best = sim.reshape(len(sim), len(ref_ids), per_ref).max(2)\n                    if i in column:\n                        best[:, column[i]] = -1.0\n                    k = min(top, best.shape[1])\n                    out[s:s + 512] = np.partition(best, -k, 1)[:, -k:].mean(1)\n                g[int(i)] = out\n                if n % 500 == 0 or n == len(todo):\n                    print(f"    {n}/{len(todo)}", flush=True)\n                    os.makedirs(os.path.dirname(path), exist_ok=True)\n                    np.savez_compressed(path, **{str(k): v for k, v in g.items()})\n        for sp in SPECIES:\n            ix = [i for i in np.where(c4.SP == sp)[0] if len(g.get(i, ()))]\n            if not ix:\n                continue\n            lo, hi = np.quantile(np.concatenate([g[i] for i in ix]), [0.5, 0.95])\n            for i in ix:\n                self.weights[int(i)] = np.clip(\n                    (hi - g[i]) / max(hi - lo, 1e-6), 0.05, 1.0).astype(np.float32)\n\n    def _descriptors_raw(self, i):\n        d, xy = super()._descriptors_raw(i)\n        w = self.weights.get(int(i), np.ones(len(d), np.float32))\n        area = (cv2.contourArea(cv2.convexHull(np.ascontiguousarray(xy)))\n                if len(xy) >= 3 else 0.0)\n        return d, xy, w, area\n\n    def verify(self, pair):\n        """(good, inliers, weighted inliers, hull fraction, rigid score, affine inliers)"""\n        a, b = pair\n        da, xa, wa, area_a = self._descriptors(a)\n        db, xb, wb, area_b = self._descriptors(b)\n        if len(da) < 4 or len(db) < 4:\n            return 0, 0, 0.0, 0.0, 0.0, 0\n        bf = cv2.BFMatcher()\n        back = np.array([m.trainIdx for m in bf.match(db, da)])\n        good = [m[0] for m in bf.knnMatch(da, db, k=2)\n                if len(m) == 2 and m[0].distance < c4.MATCH_RATIO * m[1].distance\n                and back[m[0].trainIdx] == m[0].queryIdx]\n        if len(good) < 4:\n            return len(good), 0, 0.0, 0.0, 0.0, 0\n        qi = np.array([m.queryIdx for m in good])\n        ti = np.array([m.trainIdx for m in good])\n        src, dst = np.float32(xa[qi]), np.float32(xb[ti])\n        cv2.setRNGSeed(int((int(a) * 100003 + int(b)) % (2**31 - 1)))\n        M, mask = cv2.estimateAffinePartial2D(\n            src, dst, method=cv2.RANSAC, ransacReprojThreshold=c4.RANSAC_PX)\n        if M is None or mask is None:\n            return len(good), 0, 0.0, 0.0, 0.0, 0\n        mask = mask.ravel().astype(bool)\n        inl = int(mask.sum())\n        winl = float(np.minimum(wa[qi[mask]], wb[ti[mask]]).sum())\n        hull = rigid = 0.0\n        if inl >= 5:\n            fr = [cv2.contourArea(cv2.convexHull(p[mask])) / max(area, 1e-6)\n                  for p, area in ((src, area_a), (dst, area_b))]\n            hull = float(np.clip(min(fr), 0, 1))\n            angle = abs(np.arctan2(M[1, 0], M[0, 0]))\n            logscale = abs(np.log(max(np.hypot(M[0, 0], M[1, 0]), 1e-6)))\n            rigid = float(np.exp(-angle / 0.5 - logscale / 0.5) * np.log1p(winl))\n        _, mask2 = cv2.estimateAffine2D(\n            src, dst, method=cv2.RANSAC, ransacReprojThreshold=c4.RANSAC_PX)\n        return (len(good), inl, winl, hull, rigid,\n                0 if mask2 is None else int(mask2.sum()))\n\n    def save(self):\n        os.makedirs(os.path.dirname(self.cache_path) or ".", exist_ok=True)\n        pairs = sorted(self.done)\n        np.savez_compressed(\n            self.cache_path,\n            pairs=np.array(pairs, dtype=np.int64).reshape(-1, 2),\n            values=np.array([self.done[p] for p in pairs], float).reshape(-1, 6))\n\n\ndef sift_matrices(npz_path, cache_dir, train_idx, workers):\n    cache = SiftCache3e(npz_path, os.path.join(cache_dir, "sift_pairs_3e.npz"), workers)\n    cv2.setNumThreads(1)\n    n = len(c4.X)\n    wanted = set()\n    for sp in SPECIES:                       # same neighbourhood rule as 4c\n        ix = np.where(c4.SP == sp)[0]\n        k = min(c4.NEIGHBOURS, len(ix) - 1)\n        for start in range(0, len(ix), 256):\n            q = ix[start:start + 256]\n            sim = c4.X[q] @ c4.X[ix].T\n            sim[np.arange(len(q)), np.searchsorted(ix, q)] = -9.0\n            for row, nbs in zip(q, np.argsort(-sim, 1)[:, :k]):\n                wanted.update((min(int(row), int(j)), max(int(row), int(j)))\n                              for j in ix[nbs])\n    missing = sorted(wanted - cache.done.keys())\n    print(f"  SIFT: {len(wanted)} pairs needed, {len(missing)} uncached", flush=True)\n    if missing:\n        cache.genericness(train_idx, os.path.join(cache_dir, "keypoint_generic_3e.npz"))\n        t0 = time.time()\n        with ThreadPoolExecutor(max_workers=workers) as pool:\n            for s0 in range(0, len(missing), 20000):\n                block = missing[s0:s0 + 20000]\n                for pair, value in zip(block, pool.map(cache.verify, block)):\n                    cache.done[pair] = value\n                cache.save()\n                print(f"    verified {s0 + len(block)}/{len(missing)}  "\n                      f"({time.time() - t0:.0f}s)", flush=True)\n    pp = np.array(sorted(wanted), dtype=int).reshape(-1, 2)\n    vv = np.array([cache.done[tuple(p)] for p in pp], float).reshape(-1, 6)\n    a, b = pp.T\n\n    def mat(v):\n        return sparse.coo_matrix((np.r_[v, v], (np.r_[a, b], np.r_[b, a])),\n                                 shape=(n, n)).tocsr()\n\n    c4.GOOD, c4.INL = mat(vv[:, 0].astype(int)), mat(vv[:, 1].astype(int))\n    for j, name in enumerate(EXTRA_NAMES, 2):\n        EXTRA[name] = mat(vv[:, j].astype(np.float32))\n\n\n# ═══ PLDA and the extra features ═════════════════════════════════════════════\ndef plda_scores(gi, qi, inv, n, dim=128, shrink=0.1, n_cap=4):\n    """Two-covariance PLDA log likelihood ratio: same identity vs unrelated animal."""\n    allx = c4.X[np.r_[gi, qi]]\n    d = min(dim, len(allx) - 1, allx.shape[1])\n    z = PCA(d, svd_solver="randomized", random_state=0).fit(allx).transform(allx)\n    z = c4.unit(z - z[:len(gi)].mean(0)) * np.sqrt(d)\n    zg, zq = z[:len(gi)], z[len(gi):]\n    count = np.bincount(inv, minlength=n).astype(float)\n    means = np.zeros((n, d))\n    np.add.at(means, inv, zg)\n    means /= count[:, None]\n    res = zg - means[inv]\n    Sw = res.T @ res / max(len(gi) - n, 1)\n    Sw = (1 - shrink) * Sw + shrink * (np.trace(Sw) / d + 1e-6) * np.eye(d)\n    mu = means - means.mean(0)\n    psi, V = eigh(mu.T @ mu / max(n - 1, 1), Sw)          # V\' Sw V = I\n    psi = np.maximum(psi - np.mean(1 / count), 1e-4)       # noise in the class means\n    centre = zg.mean(0)\n    uq, ug = (zq - centre) @ V, (means - centre) @ V\n    ne = np.minimum(count, n_cap)[:, None]                 # images are not independent\n    pm = ne * psi / (ne * psi + 1) * ug\n    pv = 1 + psi / (ne * psi + 1)\n    same = -0.5 * ((uq ** 2) @ (1 / pv).T - 2 * uq @ (pm / pv).T\n                   + (pm ** 2 / pv + np.log(pv)).sum(1))\n    diff = -0.5 * ((uq ** 2) @ (1 / (1 + psi)) + np.log(1 + psi).sum())\n    return same - diff[:, None]\n\n\ndef species_features(gi, qi, sp):\n    q, cd = _base_species_features(gi, qi, sp)\n    ids, inv = np.unique(c4.Y[gi], return_inverse=True)\n    n = len(ids)\n    r, c = cd["q"].to_numpy(), cd["c"].to_numpy()\n\n    def add(name, M):\n        top = -np.sort(-M, 1)[:, :2]\n        cd[f"{name}_v"] = M[r, c]\n        cd[f"{name}_gap"] = M[r, c] - top[r, 0]\n        cd[f"{name}_rk"] = (-M).argsort(1).argsort(1)[r, c]\n        q[f"{name}_top"] = top[:, 0]\n        q[f"{name}_m12"] = top[:, 0] - top[:, 1]\n\n    for name in EXTRA_NAMES:\n        S = EXTRA[name][qi][:, gi].toarray().astype(np.float32)\n        best, top2 = c4.per_identity(S, inv, n)\n        add(name, best)\n        cd[f"{name}_top2"] = top2[r, c]\n    # Raw likelihood ratios change scale with gallery size (sim vs full train), so\n    # they are standardised per query and absolute top values are not used.\n    L = plda_scores(gi, qi, inv, n)\n    L = (L - L.mean(1, keepdims=True)) / np.maximum(L.std(1, keepdims=True), 1e-9)\n    add("plda", L)\n    del q["plda_top"]\n    return q, cd\n\n\nc4.species_features = species_features\n\n\n# ═══ simulation that follows the test viewpoint mix ══════════════════════════\ndef viewpoint_weights(train_idx, test_idx):\n    """Per image: test share / train share of its (species, viewpoint), clipped."""\n    w = np.ones(len(c4.X))\n    for sp in SPECIES:\n        tr = train_idx[c4.SP[train_idx] == sp]\n        te = test_idx[c4.SP[test_idx] == sp]\n        if not len(tr) or not len(te):\n            continue\n        for v in np.unique(c4.VIEW[tr]):\n            ratio = np.mean(c4.VIEW[te] == v) / np.mean(c4.VIEW[tr] == v)\n            w[tr[c4.VIEW[tr] == v]] = np.clip(ratio, 0.35, 3.0)\n    return w\n\n\ndef simulate(seed, train_idx, view_w=None):\n    """4c\'s open-set episode; held-out images are drawn in proportion to view_w."""\n    if view_w is None:\n        return c4.simulate(seed, train_idx)\n    rng = np.random.default_rng(seed)\n    link = c4.REGIMES[seed % len(c4.REGIMES)]\n    gallery, known, unknown = [], [], []\n    for sp in SPECIES:\n        p_unknown = rng.uniform(0.2, 0.35)\n        frac = rng.uniform(*c4.FRAC)\n        cap = int(rng.integers(12, 33))\n        ix = train_idx[c4.SP[train_idx] == sp]\n        names = np.array(sorted(set(c4.Y[ix])))\n        rng.shuffle(names)\n        new_set = set(names[:int(round(p_unknown * len(names)))])\n        for name in names:\n            rows = ix[c4.Y[ix] == name].copy()\n            rng.shuffle(rows)\n            if name in new_set:\n                keys = rng.random(len(rows)) ** (1 / view_w[rows])\n                unknown.extend(rows[np.argsort(-keys)][:cap].tolist())\n            elif len(rows) < 2:\n                gallery.extend(rows.tolist())\n            else:\n                n_q = min(max(int(round(frac * len(rows))), 1), cap, len(rows) - 1)\n                groups = c4.duplicate_groups(rows, link)\n                uniq = np.unique(groups)\n                gw = np.array([view_w[rows[groups == g]].mean() for g in uniq])\n                held = []\n                for g in uniq[np.argsort(-rng.random(len(uniq)) ** (1 / gw))]:\n                    members = rows[groups == g].tolist()\n                    if len(held) < n_q and len(held) + len(members) < len(rows):\n                        held.extend(members)\n                known.extend(held[:cap])\n                held_set = set(held)\n                gallery.extend(i for i in rows if i not in held_set)\n    gi = np.array(gallery, dtype=int)\n    qi = np.array(known + unknown, dtype=int)\n    frames = c4.build_frames(gi, qi)\n    truth = np.where(np.isin(frames["order"], unknown), NEW, c4.Y[frames["order"]])\n    frames["truth"] = truth\n    frames["q"]["unk"] = (truth == NEW).astype(int)\n    frames["c"]["y"] = (frames["c"].ident.to_numpy(dtype=object) ==\n                        truth[frames["c"].q.to_numpy()]).astype(int)\n    return frames\n\n\ndef adversarial(sims, final):\n    """Can a classifier tell real test queries from simulated ones?  Returns the AUC,\n    the most telling features and a weight per simulated query."""\n    cols = [x for x in c4.columns(sims[0])[0] if x in final["q"].columns]\n    A = pd.concat([s["q"][cols] for s in sims], ignore_index=True).astype(float)\n    B = final["q"][cols].astype(float).reset_index(drop=True)\n    x = pd.concat([A, B], ignore_index=True)\n    y = np.r_[np.zeros(len(A)), np.ones(len(B))]\n    group = np.r_[np.concatenate([s["order"] for s in sims]), final["order"]]\n    p, imp = np.zeros(len(y)), np.zeros(len(cols))\n    for tr, te in GroupKFold(5).split(x, y, group):      # repeated images stay together\n        m = lgb.LGBMClassifier(n_estimators=150, num_leaves=7, learning_rate=0.05,\n                               min_child_samples=100, reg_lambda=5.0, verbose=-1)\n        m.fit(x.iloc[tr], y[tr])\n        p[te] = m.predict_proba(x.iloc[te])[:, 1]\n        imp += m.booster_.feature_importance("gain")\n    ps = np.clip(p[:len(A)], 1e-4, 1 - 1e-4)\n    w = np.clip(np.sqrt(ps / (1 - ps) * len(A) / len(B)), 0.25, 4.0)\n    top = [(cols[i], imp[i] / max(imp.sum(), 1e-9)) for i in np.argsort(-imp)[:8]]\n    return roc_auc_score(y, p), top, w / w.mean()\n\n\n# ═══ identity-disjoint training ══════════════════════════════════════════════\ndef identity_folds(train_idx, n_folds, seed):\n    """One fold per animal, balanced by image count within each species."""\n    rng = np.random.default_rng(seed)\n    fold = {}\n    for sp in SPECIES:\n        names, counts = np.unique(c4.Y[train_idx[c4.SP[train_idx] == sp]],\n                                  return_counts=True)\n        load = np.zeros(n_folds)\n        for i in np.lexsort((rng.random(len(names)), -counts)):\n            f = int(load.argmin())\n            fold[names[i]] = f\n            load[f] += counts[i]\n    return fold\n\n\ndef fit_models(Q, C, unk, y, qmask, w):\n    """4c\'s three bagged classifiers, trained on the queries selected by qmask."""\n    K = c4.K\n    cmask = np.repeat(qmask, K)\n    kmask = np.repeat(qmask & (unk == 0), K)\n    wc = np.repeat(w, K)\n\n    def bag(params, x, target, weight):\n        base = params.get("random_state", 0) or 0\n        return c4.Bag([lgb.LGBMClassifier(**{**params, "random_state": base + 101 * k})\n                       .fit(x, target, sample_weight=weight) for k in range(c4.SEEDS)])\n\n    det = {**c4.PARAMS, "n_estimators": 300,\n           "num_leaves": min(15, c4.PARAMS["num_leaves"])}\n    return (bag(det, Q[qmask], unk[qmask], w[qmask]),\n            bag(c4.PARAMS, C[kmask], y[kmask], wc[kmask]),\n            bag(c4.PARAMS, C[cmask], y[cmask], wc[cmask]))\n\n\ndef stack(sims, qfold, weights, label, rounds=None, keep_models=True):\n    """Multi-round stacking with animal-disjoint folds.  weights=None means uniform."""\n    rounds = rounds or c4.ROUNDS\n    sizes = [len(s["q"]) for s in sims]\n    cuts = np.cumsum(sizes)[:-1]\n    fold = np.concatenate(qfold)\n    w = np.ones(len(fold)) if weights is None else weights\n    round_models = []\n    for rnd in range(1, rounds + 1):\n        qf, cf = c4.columns(sims[0])\n        Q = pd.concat([s["q"][qf] for s in sims], ignore_index=True).astype(np.float32)\n        C = pd.concat([c4.candidate_matrix(s, qf, cf) for s in sims],\n                      ignore_index=True).astype(np.float32)\n        unk = np.concatenate([s["q"].unk.to_numpy() for s in sims])\n        y = np.concatenate([s["c"].y.to_numpy() for s in sims])\n        res = {k: np.zeros((len(fold), c4.K)) for k in ("raw", "pc")}\n        res.update(pn=np.zeros(len(fold)), pb=np.zeros(len(fold)))\n        for f in np.unique(fold):\n            test = fold == f\n            ctest = np.repeat(test, c4.K)\n            det, closed, anyq = fit_models(Q, C, unk, y, ~test, w)\n            raw = anyq.predict_proba(C[ctest])[:, 1].reshape(-1, c4.K)\n            within = closed.predict_proba(C[ctest])[:, 1].reshape(-1, c4.K)\n            res["raw"][test] = raw\n            res["pc"][test] = within / np.maximum(within.sum(1, keepdims=True), 1e-9)\n            res["pn"][test] = det.predict_proba(Q[test])[:, 1]\n            res["pb"][test] = (res["pn"][test] + np.clip(1 - raw.sum(1), 0, 1)) / 2\n        if keep_models:\n            round_models.append((*fit_models(Q, C, unk, y, np.ones(len(fold), bool), w),\n                                 (qf, cf)))\n        oof = [{k: v for k, v in zip(res, vals)} for vals in\n               zip(*[np.split(res[k], cuts) for k in res])]\n        ap = np.concatenate([ap_of(s, fusion(s, r)) for s, r in zip(sims, oof)])\n        print(f"  [{label}] round {rnd}: MAP@5 = {ap.mean():.4f}", flush=True)\n        if rnd < rounds:\n            sims = [c4.augment(s, r) for s, r in zip(sims, oof)]\n    return dict(models=round_models, sims=sims, oof=oof, ap=ap)\n\n\n# ═══ scores, the single ranker and the comparison ════════════════════════════\ndef fusion(s, r):\n    """4c\'s scores over K candidates + new_individual (rows sum to 1)."""\n    return np.concatenate([(1 - r["pb"])[:, None] * r["pc"], r["pb"][:, None]], 1)\n\n\ndef top5(s, scores):\n    nq = len(s["q"])\n    labels = np.concatenate([s["c"].ident.to_numpy(dtype=object).reshape(nq, c4.K),\n                             np.full((nq, 1), NEW, dtype=object)], 1)\n    return np.take_along_axis(labels, np.argsort(-scores, 1, kind="stable")[:, :5], 1)\n\n\ndef ap_of(s, scores):\n    hit = top5(s, scores) == s["truth"][:, None]\n    return (hit / np.arange(1, 6)).sum(1)\n\n\ndef rank_matrix(s, qf, cf):\n    """(queries x (K+1), features): K candidate rows then one new_individual row."""\n    nq, K = len(s["q"]), c4.K\n    C = s["c"][cf].to_numpy(np.float32).reshape(nq, K, len(cf))\n    C = np.concatenate([C, np.full((nq, 1, len(cf)), np.nan, np.float32)], 1)\n    Q = np.repeat(s["q"][qf].to_numpy(np.float32)[:, None, :], K + 1, 1)\n    flag = np.zeros((nq, K + 1, 1), np.float32)\n    flag[:, K] = 1\n    return np.concatenate([C, Q, flag], 2).reshape(nq * (K + 1), -1)\n\n\ndef fit_ranker(X, rel, qmask, w):\n    """rel: (queries, K+1) relevance.  Queries with no correct row are skipped."""\n    use = qmask & (rel.sum(1) > 0)\n    rows = np.repeat(use, rel.shape[1])\n    params = {k: v for k, v in c4.PARAMS.items() if k != "objective"}\n    return [lgb.LGBMRanker(objective="lambdarank", lambdarank_truncation_level=8,\n                           **{**params, "random_state": 11 + 101 * k})\n            .fit(X[rows], rel[use].ravel(), group=np.full(use.sum(), rel.shape[1]),\n                 sample_weight=np.repeat(w[use], rel.shape[1]))\n            for k in range(c4.SEEDS)]\n\n\ndef ranker_scores(models, X):\n    z = np.mean([m.predict(X, raw_score=True) for m in models], 0).reshape(-1, c4.K + 1)\n    e = np.exp(z - z.max(1, keepdims=True))\n    return e / e.sum(1, keepdims=True)\n\n\ndef final_layer(run, qfold, weights):\n    """Out-of-fold ranker on top of the last round; returns scores for three choices."""\n    sims = [c4.augment(s, r) for s, r in zip(run["sims"], run["oof"])]\n    qf, cf = c4.columns(sims[0])\n    X = np.concatenate([rank_matrix(s, qf, cf) for s in sims])\n    rel = np.concatenate([np.concatenate(\n        [s["c"].y.to_numpy().reshape(-1, c4.K), s["q"].unk.to_numpy()[:, None]], 1)\n        for s in sims])\n    fold = np.concatenate(qfold)\n    w = np.ones(len(fold)) if weights is None else weights\n    scores = np.zeros((len(fold), c4.K + 1))\n    for f in np.unique(fold):\n        test = fold == f\n        scores[test] = ranker_scores(fit_ranker(X, rel, ~test, w),\n                                     X[np.repeat(test, c4.K + 1)])\n    parts = np.split(scores, np.cumsum([len(s["q"]) for s in sims])[:-1])\n    options = {"fusion": [fusion(s, r) for s, r in zip(sims, run["oof"])],\n               "ranker": parts}\n    options["blend"] = [(a + b) / 2 for a, b in zip(options["fusion"], parts)]\n    ap = {k: np.concatenate([ap_of(s, m) for s, m in zip(sims, v)])\n          for k, v in options.items()}\n    full = (fit_ranker(X, rel, np.ones(len(fold), bool), w), qf, cf)\n    return ap, full, options\n\n\ndef bootstrap(diff, animals, weights=None, reps=2000, seed=0):\n    """Mean difference and 95% interval, resampling whole animals."""\n    w = np.ones(len(diff)) if weights is None else weights\n    _, g = np.unique(animals, return_inverse=True)\n    num, den = np.bincount(g, diff * w), np.bincount(g, w)\n    rng = np.random.default_rng(seed)\n    draws = rng.integers(0, len(num), (reps, len(num)))\n    boot = num[draws].sum(1) / den[draws].sum(1)\n    return num.sum() / den.sum(), np.quantile(boot, [0.025, 0.975])\n\n\ndef prior_em(p, train_prior, iters=200):\n    """Saerens et al. 2002: re-estimate the new-animal share from unlabelled scores."""\n    pi = train_prior\n    for _ in range(iters):\n        a = p * pi / train_prior\n        post = a / (a + (1 - p) * (1 - pi) / (1 - train_prior))\n        pi = float(np.clip(post.mean(), 1e-3, 1 - 1e-3))\n    return pi, post\n\n\nRETUNE_GRID = [(31, 40, 2.0), (15, 40, 2.0), (15, 100, 5.0),\n               (31, 100, 5.0), (7, 100, 10.0), (63, 40, 2.0)]\n\n\n'
exec(compile(_MODEL3E_SOURCE, "<embedded Model 3e>", "exec"), globals())

# Match 3e defaults.
c4.K = K_CANDIDATES
c4.ROUNDS = N_ROUNDS
c4.NEIGHBOURS = N_NEIGHBOURS
c4.SEEDS = N_SEEDS
c4.PARAMS.update(n_estimators=N_TREES, random_state=RANDOM_SEED)
print("Embedded Model 3e definitions loaded; no Dewang-module import used.")

## Load data and build fixed episodes

The simulation seeds and animal folds are fixed for all compared settings. Test labels are never used. Test metadata and unlabelled features are used only for the same viewpoint-mix adaptation that Model 3e uses.

In [ ]:
t0 = time.time()
train_df, test_df, master_df, train_idx, test_idx = c4.load_data(str(ROOT))
assert (PROC / "sift_features.npz").exists(), "Missing Processed_Data/sift_features.npz"
print(f"Train={len(train_df):,}; test={len(test_df):,}; identities={train_df.identity.nunique():,}")

# Model 3e richer SIFT features, persisted inside this experiment folder.
sift_matrices(str(PROC / "sift_features.npz"), str(CACHE), train_idx, N_WORKERS)
view_w = viewpoint_weights(train_idx, test_idx)
sims = [simulate(RANDOM_SEED + i, train_idx, view_w) for i in range(N_SIMS)]
print(f"Built {len(sims)} fixed episodes; mean candidate recall@{c4.K}: "
      f"{np.mean([c4.candidate_recall(s) for s in sims]):.4f}; elapsed={time.time()-t0:.1f}s")

fold_map = identity_folds(train_idx, N_FOLDS, RANDOM_SEED)
qfold = [np.array([fold_map[c4.Y[i]] for i in s["order"]]) for s in sims]
animals = c4.Y[np.concatenate([s["order"] for s in sims])].astype(str)
# Test feature frame is unlabelled; it is needed for the existing 3e shift weights.
final_test = c4.build_frames(train_idx, test_idx)

## Train Model 3e and collect out-of-fold scores

This reproduces the 3e validation procedure: animal-disjoint folds for the meta-models, three rounds of transductive stacking, then a LambdaRank final layer. The method retains 3e’s previously documented caveat that neighbour-stack feature generation is not fully nested.

In [ ]:
auc, shift_features, adv_weights = adversarial(sims, final_test)
print(f"Test-vs-simulation classifier AUC={auc:.3f}; strongest shift features={shift_features[:5]}")

# Match 3e: keep covariate weights only when they improve weighted stack OOF MAP.
run = stack(sims, qfold, None, "Model 3e plain", rounds=N_ROUNDS, keep_models=True)
weighted_run = stack(sims, qfold, adv_weights, "Model 3e test-shift weighted", rounds=N_ROUNDS, keep_models=True)
plain_weighted_score = float(np.average(run["ap"], weights=adv_weights))
adapt_weighted_score = float(np.average(weighted_run["ap"], weights=adv_weights))
if adapt_weighted_score > plain_weighted_score:
    run, training_weights = weighted_run, adv_weights
else:
    training_weights = None
del weighted_run
metric_weights = adv_weights
print(f"Selected stack: {'weighted' if training_weights is not None else 'plain'} "
      f"(weighted OOF {plain_weighted_score:.4f} vs {adapt_weighted_score:.4f})")
layer_scores, ranker, fold_options = final_layer(run, qfold, training_weights)
score_layer = lambda values: float(np.average(values, weights=metric_weights))
base_choice = max(layer_scores, key=lambda k: score_layer(layer_scores[k]))
print("3e final layer MAP@5 (weighted):", {k: round(score_layer(v), 6) for k, v in layer_scores.items()})
print("Baseline selection:", base_choice)

## Species-aware unknown-prior correction

EM estimates a target-set unknown share from the Model 3e detector probability `pb`. The estimated-vs-training prior ratio changes only the `new_individual` score; known identity order stays the same. A strength of 0 is the unadjusted baseline, while 1 applies the full odds correction. Global adjustment estimates one prior across both species; species-aware adjustment estimates one per species and per episode.

For a fair paired comparison, all settings use the same OOF predictions and same baseline ranker/fusion choice. Bootstrap resampling is by animal, not by individual query.

In [ ]:
def _em_prior(p, train_prior, iters=200):
    p = np.clip(np.asarray(p, float), 1e-5, 1 - 1e-5)
    pi0 = float(np.clip(train_prior, 1e-4, 1 - 1e-4))
    pi = pi0
    for _ in range(iters):
        post = (p * pi / pi0) / (p * pi / pi0 + (1-p) * (1-pi) / (1-pi0))
        new_pi = float(np.clip(post.mean(), 1e-4, 1 - 1e-4))
        if abs(new_pi - pi) < 1e-8:
            pi = new_pi
            break
        pi = new_pi
    post = (p * pi / pi0) / (p * pi / pi0 + (1-p) * (1-pi) / (1-pi0))
    return pi, post

def _prior_factor(p, pi0, pi1):
    p = np.clip(p, 1e-5, 1 - 1e-5)
    post = (p * pi1 / pi0) / (p * pi1 / pi0 + (1-p) * (1-pi1) / (1-pi0))
    return np.clip((post / (1-post)) / (p / (1-p)), 0.05, 20.0)

def adjust_scores(scores, pb, species, pi_by_species, mode, strength):
    out = np.asarray(scores, float).copy()
    species = np.asarray(species)
    if mode == "global":
        pi0 = pi_by_species["__global__"]
        pi1, _ = _em_prior(pb, pi0)
        out[:, -1] *= _prior_factor(pb, pi0, pi1) ** strength
        out /= np.maximum(out.sum(1, keepdims=True), 1e-12)
        return out
    if mode != "species":
        return out
    for sp in ("hyena", "leopard"):
        mask = species == sp
        if not mask.any():
            continue
        pi0 = pi_by_species[sp]
        pi1, _ = _em_prior(pb[mask], pi0)
        out[mask, -1] *= _prior_factor(pb[mask], pi0, pi1) ** strength
        out[mask] /= np.maximum(out[mask].sum(1, keepdims=True), 1e-12)
    return out

cal_sims = run["sims"]
base_scores = np.concatenate(fold_options[base_choice])
truth = np.concatenate([s["truth"] for s in cal_sims])
sp_all = np.concatenate([c4.SP[s["order"]] for s in cal_sims])
pb_all = np.concatenate([r["pb"] for r in run["oof"]])
unk_all = np.concatenate([s["q"].unk.to_numpy() for s in run["sims"]])
prior_weights = np.ones(len(unk_all)) if training_weights is None else training_weights
species_all = np.concatenate([c4.SP[s["order"]] for s in run["sims"]])
pi_species = {sp: float(np.average(unk_all[species_all == sp],
                                   weights=prior_weights[species_all == sp]))
              for sp in ("hyena", "leopard")}
pi_species["__global__"] = float(np.average(unk_all, weights=prior_weights))
print("Training unknown priors:", pi_species)

def _map_scores(score_matrix, truth_values):
    preds = np.take_along_axis(
        np.concatenate([np.concatenate([s["c"].ident.to_numpy(dtype=object).reshape(-1, c4.K),
                                        np.full((len(s["q"]), 1), NEW, dtype=object)], 1)
                        for s in cal_sims]),
        np.argsort(-score_matrix, axis=1, kind="stable")[:, :5], axis=1)
    hit = preds == truth_values[:, None]
    return (hit / np.arange(1, 6)).sum(1)

def _animal_bootstrap(delta, animal_ids, weights, reps=3000, seed=2026):
    _, group = np.unique(animal_ids, return_inverse=True)
    sums = np.bincount(group, weights=delta * weights)
    mass = np.bincount(group, weights=weights)
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(sums), (reps, len(sums)))
    vals = sums[draws].sum(1) / np.maximum(mass[draws].sum(1), 1e-12)
    return float(np.average(delta, weights=weights)), np.quantile(vals, [0.025, 0.975])

base_ap = _map_scores(base_scores, truth)
results = []
for method in ("none", "global", "species"):
    for strength in ((0.0,) if method == "none" else (0.25, 0.5, 0.75, 1.0)):
        adjusted = base_scores if method == "none" else np.vstack([
            adjust_scores(base_scores[sum(len(s["q"]) for s in cal_sims[:i]):sum(len(s["q"]) for s in cal_sims[:i+1])],
                          run["oof"][i]["pb"], c4.SP[cal_sims[i]["order"]], pi_species, method, strength)
            for i in range(len(cal_sims))])
        ap = _map_scores(adjusted, truth)
        delta, ci = _animal_bootstrap(ap - base_ap, animals, metric_weights)
        results.append({"method": method, "strength": strength,
                        "map5_weighted": float(np.average(ap, weights=metric_weights)),
                        "map5_unweighted": float(ap.mean()),
                        "delta_weighted": delta, "ci_low": float(ci[0]), "ci_high": float(ci[1])})
results_df = pd.DataFrame(results).sort_values("map5_weighted", ascending=False).reset_index(drop=True)
display(results_df)

## Decision and submission

The best row above is an exploratory choice on the same animal-disjoint OOF validation used for model comparison. Treat its bootstrap interval as uncertainty about the average animal-level difference on this simulation protocol; it does not measure leaderboard uncertainty. The default is to stop before test inference. Set `RUN_TEST_INFERENCE = True` in the configuration cell only after reviewing the table.

In [ ]:
if RUN_TEST_INFERENCE:
    # Select the prior mode/strength with the highest OOF MAP; this uses no hidden test labels.
    winner = results_df.iloc[0]
    selected_mode = str(winner.method)
    selected_strength = float(winner.strength)
    selected_ranker = ranker
    test_frame = final_test
    last_result = None
    for rnd, models in enumerate(run["models"], 1):
        last_result = c4.predict(models, test_frame)
        if rnd < N_ROUNDS:
            test_frame = c4.augment(test_frame, last_result)
    # Add the final round predictions, matching the feature schema used to train the ranker.
    test_frame = c4.augment(test_frame, last_result)
    # final_layer returns (ranker_models, query_features, candidate_features).
    rank_models, rank_qf, rank_cf = ranker
    rank_test = ranker_scores(rank_models, rank_matrix(test_frame, rank_qf, rank_cf))
    fusion_test = fusion(test_frame, last_result)
    test_options = {"fusion": fusion_test, "ranker": rank_test, "blend": (fusion_test + rank_test) / 2}
    chosen_scores = test_options[base_choice]
    if selected_mode != "none":
        species_test = c4.SP[test_frame["order"]]
        chosen_scores = adjust_scores(chosen_scores, last_result["pb"], species_test,
                                      pi_species, selected_mode, selected_strength)
    preds = top5(test_frame, chosen_scores)
    sub = pd.DataFrame({"id": c4.IDS[test_frame["order"]],
                        "predictions": [" ".join(row) for row in preds]})
    sample_path = PROC / "sample_submission.csv"
    if not sample_path.exists():
        sample_path = ROOT / "DATA/reid_student_data_main/sample_submission.csv"
    if sample_path.exists():
        sample = pd.read_csv(sample_path)
        sub = sample[["id"]].merge(sub, on="id", how="left", validate="one_to_one")
    c4.validate_submission(sub, test_df, train_df, str(sample_path))
    output = NOTEBOOK_DIR / "submission_model3e_plus.csv"
    sub.to_csv(output, index=False)
    print(f"Saved {output}; baseline layer={base_choice}, prior mode={selected_mode}, strength={selected_strength}")
else:
    print("Test inference is disabled. Review validation before enabling it.")